# DocLens: LoRA fine-tune for invoice extraction (F8)

Fine-tunes Qwen2-VL-2B (4-bit QLoRA) on ~1,500 synthetic Japanese invoices, then scores the base model and the adapter on the held-out
synthetic set with the same metrics. **Status: written but not executed**; expect to fix version-specific details on the first run.

Settings: *GPU T4 x2*, Internet on. Training uses one GPU; the second serves the evaluation.
Outputs: `eval/results/direct-base.json`, `eval/results/direct-lora.json` and the adapter in `train/adapter/`.
Report the training time, the GPU and the dataset card (`train/data`: how many examples, seed, the eval seed it never touches).

In [ ]:
import os, subprocess, time, urllib.request, json
REPO_URL = "https://github.com/shizen-57/PaperGraph.git"
!git clone $REPO_URL doclens_repo
%cd doclens_repo/doclens
!pip install -q -r requirements.txt faker augraphy playwright rapidfuzz apted lxml
!pip install -q "transformers>=4.45" peft bitsandbytes accelerate qwen-vl-utils vllm
!playwright install chromium

## 1. Data: synthetic invoices (seed 1; the eval set is seed 0)

In [ ]:
t0 = time.time()
!python -m train.build_dataset --n 1500 --seed 1 --out train/data
print("dataset minutes:", round((time.time() - t0) / 60, 1))

## 2. Train

In [ ]:
t0 = time.time()
!python -m train.lora_vlm --data train/data --out train/adapter
print("training minutes:", round((time.time() - t0) / 60, 1))

## 3. Evaluate before and after
vLLM serves the base model with the adapter attached as a named LoRA, so one server answers both: the base model name for "before",
`invoice-lora` for "after".

In [ ]:
log = open("vllm_vlm.log", "w")
server = subprocess.Popen(["python", "-m", "vllm.entrypoints.openai.api_server", "--model", "Qwen/Qwen2-VL-2B-Instruct", "--dtype", "half",
                           "--max-model-len", "8192", "--enable-lora", "--max-lora-rank", "16",
                           "--lora-modules", "invoice-lora=train/adapter", "--port", "8002"],
                          stdout=log, stderr=log, env=dict(os.environ, CUDA_VISIBLE_DEVICES="1"))
for _ in range(180):
    try:
        urllib.request.urlopen("http://localhost:8002/health"); break
    except Exception:
        time.sleep(10)
os.environ["VLM_BASE_URL"] = "http://localhost:8002/v1"
os.environ["LLM_API_KEY"] = "none"
!python eval/run_direct.py --model Qwen/Qwen2-VL-2B-Instruct --out eval/results/direct-base.json --note "before: base model, zero-shot"
!python eval/run_direct.py --model invoice-lora --out eval/results/direct-lora.json --note "after: QLoRA r=16, 1 epoch, ~1,500 synthetic invoices"
server.terminate()

In [ ]:
!python eval/render_tables.py
!zip -qr /kaggle/working/lora_results.zip eval/results train/adapter